<a href="https://colab.research.google.com/github/eteitelbaum/code-satp/blob/main/training-and-inference-notebooks/training_location_extraction_t5_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --upgrade transformers


In [ ]:
!pip install datasets evaluate rouge-score

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load your dataset
data = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_matched.csv', dtype=str)

def prepare_data(df):
    inputs = [f"Extract the location of the incident: {summary}" for summary in df['incident_summary']]
    targets = df['Matched_Location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

print(dataset)



In [ ]:
from transformers import T5Tokenizer

# Initialize the tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# # Tokenization function
# def preprocess_function(examples):
#     # Tokenize the input and target text
#     model_inputs = tokenizer(
#         examples['input_text'],
#         max_length=512,
#         truncation=True,
#         padding="max_length"
#     )
#     labels = tokenizer(
#         examples['target_text'],
#         max_length=64,
#         truncation=True,
#         padding="max_length"
#     ).input_ids

#     # # Replace padding token id's in the labels with -100
#     # labels = [
#     #     [(l if l != tokenizer.pad_token_id else -100) for l in label]
#     #     for label in labels
#     # ]
#     model_inputs["labels"] = labels
#     return model_inputs

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_datasets = dataset.map(preprocess_function, batched=True)


In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calculate exact match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    accuracy = sum(exact_matches) / len(exact_matches)

    return {"exact_match": accuracy * 100}  # Return as percentage


In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer
import evaluate

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)

# Define compute metrics
# rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels as we can't decode them
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    # rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    # Compute BLEU scores
    bleu_result = bleu.compute(predictions=decoded_preds, references=[[ref] for ref in decoded_labels])

    # Combine results
    metrics = {
        # "rouge1": rouge_result["rouge1"].mid.fmeasure * 100,
        # "rouge2": rouge_result["rouge2"].mid.fmeasure * 100,
        # "rougeL": rouge_result["rougeL"].mid.fmeasure * 100,
        "bleu": bleu_result["bleu"] * 100,
    }
    return metrics

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for checkpoints and logs
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=5e-5,             # Learning rate
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=4,   # Batch size for evaluation
    num_train_epochs=3,             # Number of training epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_strategy="epoch",          # Save the model at each epoch
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=100,              # Log every 100 steps
    load_best_model_at_end=True, # Load the best model at the end of training
    report_to='none'
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    # compute_metrics=compute_metrics
)


In [ ]:

# Train the model
trainer.train()


In [ ]:
# Save the model and tokenizer
model.save_pretrained('/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model')
tokenizer.save_pretrained('/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model')


In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test Set Evaluation Results: {eval_results}")


In [ ]:
def predict_location(summary):
    # Prepare the input text
    input_text = f"Extract the location of the incident: {summary}"
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

model.to('cpu')
# Example usage
incident_summary = "Three Communist Party of India-Maoist (CPI-Maoist) cadres, identified as Madvi Bhima (52), Madkam Hidma alias Sai Denga (33), and Paadam Ayate (24), surrendered in Sukma District in the Bastar division of Chhattisgarh on September 19, reports devdiscourse.com. Bhima was the ?president? of the Nilamadgu Revolutionary People?s Committee (RPC) Dandakaranya Adivasi Kisan Mazdoor Sangthan (DAKMS) unit of the Maoist group and carried a INR 100,000 reward. Hidma served as ?vice-president?, while Ayate was the ?head? of Palachalma RPC 'Jantana Sarkar' (people's government of the Maoists), each also carrying a INR 100,000 reward. The trio cited their disillusionment with Maoist atrocities and were impressed by the state's Naxalite [Left Wing Extremist, LWE] elimination policy and welfare schemes as reasons for their surrender."
predicted_locations = predict_location(incident_summary)
print(f"Predicted Locations: {predicted_locations}")


In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model_path = '/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'Using device: {device}')


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load your dataset
data = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_matched.csv', dtype=str)

def prepare_data(df):
    inputs = [f"Extract the location of the incident: {summary}" for summary in df['incident_summary']]
    targets = df['Matched_Location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_dataset_test = dataset["test"].map(preprocess_function, batched=True)
# Set format for PyTorch
tokenized_dataset_test.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)


In [ ]:
import numpy as np

def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Calculate exact match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    accuracy = sum(exact_matches) / len(exact_matches)

    return {"exact_match": accuracy * 100}  # Return percentage


In [ ]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())  # Move labels to CPU

    # Compute metrics
    metrics = compute_metrics(all_predictions, all_labels)
    return metrics


In [ ]:
from torch.utils.data import DataLoader

# Convert the test dataset to a PyTorch DataLoader
test_loader = DataLoader(tokenized_dataset_test, batch_size=8)


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


In [ ]:
!pip install evaluate nltk rouge_score

In [ ]:
import evaluate

# Load BLEU and ROUGE evaluators
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Format labels for BLEU (it expects a list of lists for references)
    formatted_labels = [[label] for label in decoded_labels]

    # Compute BLEU
    bleu = bleu_metric.compute(predictions=decoded_preds, references=formatted_labels)

    # Compute ROUGE
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Combine and return metrics
    metrics = {
        "bleu": bleu["bleu"] * 100,  # Convert to percentage
        "rouge1": rouge["rouge1"] * 100,
        "rouge2": rouge["rouge2"] * 100,
        "rougeL": rouge["rougeL"] * 100,
    }
    return metrics


In [ ]:
def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.cpu().numpy())  # Move labels to CPU

    # Compute metrics
    metrics = compute_metrics(all_predictions, all_labels)
    return metrics


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


In [ ]:
def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Exact Match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    exact_match_score = sum(exact_matches) / len(exact_matches)

    # Recall
    recalls = []
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_components = set(pred.strip().split(", "))
        label_components = set(label.strip().split(", "))
        recall = len(pred_components & label_components) / len(label_components)
        recalls.append(recall)
    avg_recall = sum(recalls) / len(recalls)

    return {
        "exact_match": exact_match_score * 100,
        "recall": avg_recall * 100,
    }


In [ ]:
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


In [ ]:
import pandas as pd
import torch

# Initialize the model for evaluation
model.eval()

# Lists to store the results
incident_summaries = []
predicted_locations = []
actual_locations = []

# Evaluation loop
with torch.no_grad():
    for batch in test_loader:
        # Move input data to the appropriate device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        # Generate predictions
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=64
        )
        predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        labels_decoded = [
            tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
            for label in labels
        ]

        # Append data to lists
        incident_summaries.extend(
            tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        )
        predicted_locations.extend(predictions)
        actual_locations.extend(labels_decoded)

# Create a DataFrame
test_results_df = pd.DataFrame({
    "Incident Summary": incident_summaries,
    "Predicted Location": predicted_locations,
    "Actual Location": actual_locations,
})

# Save the DataFrame to a CSV file (optional)
test_results_df.to_csv('/content/drive/MyDrive/SATP_data/location_extraction_test_results.csv', index=False)

# Display the DataFrame
test_results_df.head()


In [ ]:
import requests
import pandas as pd
from gliner import GLiNER

# Load a pre-trained NER model
model = GLiNER.from_pretrained("urchade/gliner_base")

# Google Maps API key
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

# Function to extract locations using NER
def extract_locations(text):
    """Extracts location entities from the given text using GLiNER."""
    labels = ["location"]  # Focus on location entities
    entities = model.predict_entities(text, labels)
    locations = [entity["text"] for entity in entities if entity["label"] == "location"]
    return locations

# Function to get location details using Google Geocoding API
def get_location_details(locations):
    """Given a list of location names, constructs a query, calls the Google Geocoding API,
    and returns state, district, subdistrict, and town/village (if any)."""
    query = ', '.join(locations)
    params = {
        'address': query,
        'key': API_KEY
    }
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        print(f"Geocoding API error: {data['status']}")
        return None

    # Parse the address components
    results = data.get('results', [])
    if not results:
        print("No results found.")
        return None

    # Take the first result
    address_components = results[0]['address_components']

    # Initialize components
    state = district = subdistrict = town_village = None

    # Map Google address types to desired components
    for component in address_components:
        types = component['types']
        if 'administrative_area_level_1' in types:
            state = component['long_name']
        elif 'administrative_area_level_2' in types:
            district = component['long_name']
        elif 'administrative_area_level_3' in types:
            subdistrict = component['long_name']
        elif 'locality' in types:
            town_village = component['long_name']
        elif 'sublocality' in types and not town_village:
            town_village = component['long_name']

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village
    }

# Load the incident data
satp_dat = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_id.csv', dtype=str)
satp_data = satp_dat.head(100).copy()

satp_data['extracted_state'] = None
satp_data['extracted_district'] = None
satp_data['extracted_subdistrict'] = None
satp_data['extracted_town_village'] = None


# Process each summary in the data
for index, row in satp_data.iterrows():
    summary = row['incident_summary']  # Adjust column name if necessary
    locations = extract_locations(summary)
    satp_data.at[index, 'extracted_locations'] = locations
    if locations:
        location_details = get_location_details(locations)
        if location_details:
            # Update the DataFrame with location details
            satp_data.at[index, 'extracted_state'] = location_details.get('state')
            satp_data.at[index, 'extracted_district'] = location_details.get('district')
            satp_data.at[index, 'extracted_subdistrict'] = location_details.get('subdistrict')
            satp_data.at[index, 'extracted_town_village'] = location_details.get('town_village')
    else:
        print(f"No locations found in summary at index {index}")

# Display the updated DataFrame
print(satp_data.head())
